# Spaceship Titanic — ベースライン（解説付き）

**目的:** 乗客の情報から、宇宙船事故後に**別次元へ転送されたか（Transported: True/False）**を予測する。

**このノートブックの流れ:**
1. データを読み、何が効きそうか観察する（EDA）
2. Cabin の分割や支出合計など特徴量を作る（前処理・FE）
3. RandomForest で学習し、5-fold CV で accuracy を確認
4. `output/submission.csv` を作って Kaggle に提出


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)


## 1. データ読み込み

`data/` に CSV がない場合は、ターミナルで以下を実行してください。

```bash
uv run kaggle competitions download -c spaceship-titanic -p data
```


In [ ]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_path = DATA_DIR / "sample_submission.csv"

for path in (train_path, test_path, sample_path):
    if not path.exists():
        raise FileNotFoundError(f"{path} が見つかりません。上のセルの手順でデータを取得してください。")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_path)

print(f"train: {train.shape}, test: {test.shape}")
train.head()


## 2. EDA — 何が Transported と関係ありそうか？

Spaceship Titanic では **CryoSleep（冷凍睡眠）** 中の乗客や **支出が少ない乗客** が転送されにくい、といった傾向がよく見られます。

| 列 | 意味 |
|---|---|
| Transported | True/False（**予測したい正解**） |
| HomePlanet | 出身惑星（Earth / Europa / Mars） |
| CryoSleep | 冷凍睡眠中か |
| Cabin | 客室（`Deck/Num/Side` 形式） |
| Destination | 目的地 |
| Age, VIP | 年齢・VIP か |
| RoomService 等 | 船内サービスの利用額 |
| Name | 氏名（グループ分析には PassengerId を使う） |
| PassengerId | `グループ番号_グループ内番号` |


In [ ]:
train.info()
train.describe(include="all").T


In [ ]:
missing = pd.DataFrame({
    "train": train.isna().mean(),
    "test": test.isna().mean(),
}).sort_values("train", ascending=False)
missing[missing.max(axis=1) > 0]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sns.countplot(data=train, x="Transported", ax=axes[0])
axes[0].set_title("Transported")

sns.barplot(data=train, x="HomePlanet", y="Transported", ax=axes[1], errorbar=None)
axes[1].set_title("Transport rate by HomePlanet")

sns.barplot(data=train, x="CryoSleep", y="Transported", ax=axes[2], errorbar=None)
axes[2].set_title("Transport rate by CryoSleep")

plt.tight_layout()
plt.show()


In [ ]:
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
train_plot = train.copy()
train_plot["TotalSpend"] = train_plot[spend_cols].fillna(0).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(data=train_plot, x="TotalSpend", hue="Transported", kde=True, ax=axes[0], multiple="stack")
axes[0].set_title("TotalSpend distribution")

sns.barplot(data=train, x="Destination", y="Transported", ax=axes[1], errorbar=None)
axes[1].set_title("Transport rate by Destination")

plt.tight_layout()
plt.show()


## 3. 前処理・特徴量エンジニアリング

**やること:**
- **Cabin 分割** … `Deck` / `CabinNum` / `Side`（例: `B/0/P`）
- **TotalSpend** … 5 つの支出列の合計（CryoSleep 中は 0 になりやすい）
- **GroupSize** … `PassengerId` の `_` より前が同じ乗客の人数（train+test を結合して数える）
- **IsAlone** … GroupSize == 1 かどうか

数値列は **中央値**、カテゴリ列は **最頻値** で欠損補完し、HomePlanet / Destination / Deck / Side などは **One-Hot 符号化** します。


In [ ]:
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def add_group_size(train_df: pd.DataFrame, test_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    combined = pd.concat(
        [
            train_df.assign(_split="train"),
            test_df.assign(_split="test"),
        ],
        ignore_index=True,
    )
    combined["GroupId"] = combined["PassengerId"].str.split("_", n=1).str[0]
    combined["GroupSize"] = combined.groupby("GroupId")["GroupId"].transform("count")

    train_out = combined.loc[combined["_split"] == "train"].drop(columns=["_split"])
    test_out = combined.loc[combined["_split"] == "test"].drop(columns=["_split"])
    return train_out, test_out


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    cabin_parts = out["Cabin"].astype(str).str.split("/", expand=True)
    out["Deck"] = cabin_parts[0].replace("nan", np.nan)
    out["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    out["Side"] = cabin_parts[2].replace("nan", np.nan)

    out["TotalSpend"] = out[SPEND_COLS].fillna(0).sum(axis=1)
    out["IsAlone"] = (out["GroupSize"] == 1).astype(int)

    return out


train_gs, test_gs = add_group_size(train, test)
train_fe = add_features(train_gs)
test_fe = add_features(test_gs)

FEATURE_COLUMNS = [
    "HomePlanet",
    "CryoSleep",
    "Destination",
    "Age",
    "VIP",
    *SPEND_COLS,
    "TotalSpend",
    "Deck",
    "CabinNum",
    "Side",
    "GroupSize",
    "IsAlone",
]

X = train_fe[FEATURE_COLUMNS]
y = train_fe["Transported"].astype(int)
X_test = test_fe[FEATURE_COLUMNS]

X.head()


In [ ]:
numeric_features = [
    "Age",
    *SPEND_COLS,
    "TotalSpend",
    "CabinNum",
    "GroupSize",
    "IsAlone",
]
categorical_features = ["HomePlanet", "CryoSleep", "Destination", "VIP", "Deck", "Side"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model),
])


## 4. モデル学習・CV 評価

**RandomForest** を使います。決定木をたくさん作って多数決するイメージです。

- **StratifiedKFold（5 分割）** … 各 fold で Transported True/False の比率を保ったまま分割
- **CV accuracy ≈ 0.80** が目安（ランダム予測は約 0.50）

※ 訓練データ全体の正解率（train accuracy）は CV より高く出がちなので、**CV の方を信頼**してください。


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy", n_jobs=-1)

print(f"CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"fold scores: {cv_scores}")

pipeline.fit(X, y)
train_pred = pipeline.predict(X)
print(f"train accuracy: {accuracy_score(y, train_pred):.4f}")


## 5. 提出ファイル作成


In [ ]:
test_pred = pipeline.predict(X_test)

submission = sample_submission.copy()
submission["Transported"] = test_pred.astype(bool)

submission_path = OUTPUT_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"saved: {submission_path.resolve()}")
submission.head(10)


In [ ]:
# Kaggle へ提出（任意）
# !uv run kaggle competitions submit -c spaceship-titanic -f ../output/submission.csv -m "baseline rf"
